# World Bank Data Download

**Purpose:** Download macroeconomic and fiscal indicators for **all World Bank economies** (1990–2024).

We download the full global dataset here — no country filtering yet.  
Country selection happens later, once you have reviewed data coverage and finalised  
`COUNTRIES_BY_TYPE` in `config/settings.py`.

**Output:** `data/raw/world_bank_data.csv`

| WB Code | Column name | Description |
|---|---|---|
| NY.GDP.MKTP.CD | gdp_current_usd | GDP (current US$) |
| NY.GDP.MKTP.KD.ZG | gdp_growth | GDP growth (annual %) |
| NY.GDP.PCAP.CD | gdp_per_capita_usd | GDP per capita (current US$) |
| SP.POP.TOTL | population | Total population |
| GC.DOD.TOTL.GD.ZS | debt_to_gdp | Central gov debt (% GDP) |
| GC.BAL.CASH.GD.ZS | fiscal_balance_gdp | Cash surplus/deficit (% GDP) |
| GC.TAX.TOTL.GD.ZS | tax_revenue_gdp | Tax revenue (% GDP) |
| GC.XPN.TOTL.GD.ZS | gov_expenditure_gdp | Government expenditure (% GDP) |
| GC.XPN.INTP.GD.ZS | interest_payments_gdp | Interest payments (% GDP) |
| FP.CPI.TOTL.ZG | inflation_cpi | CPI inflation (annual %) |
| NY.GDP.DEFL.KD.ZG | inflation_deflator | GDP deflator inflation (annual %) |
| FR.INR.LNDP.ZS | real_interest_rate | Real interest rate (%) |
| NE.EXP.GNFS.ZS | exports_gdp | Exports of goods & services (% GDP) |
| NE.IMP.GNFS.ZS | imports_gdp | Imports of goods & services (% GDP) |
| BN.CAB.XOKA.GD.ZS | current_account_gdp | Current account balance (% GDP) |
| FI.RES.TOTL.CD | foreign_reserves_usd | Total reserves (current US$) |

## Cell 1 — Imports and project root

`find_project_root()` walks up the directory tree from wherever this notebook is stored  
until it finds the `config/` folder. This works correctly whether the notebook is in  
`src/notebooks/`, `notebooks/`, or anywhere else in the project.

In [2]:
%pip install wbgapi

In [5]:
from pathlib import Path
print(Path().resolve())

/content


In [ ]:
import sys
import time
import logging
from pathlib import Path

import pandas as pd
import wbgapi as wb


def find_project_root(start: Path = Path().resolve()) -> Path:
    for directory in [start, *start.parents]:
        if (directory / "config").is_dir():
            return directory
    raise FileNotFoundError(
        "Cannot find project root. "
        "Expected a config/ folder somewhere above this notebook."
    )

sys.path.append(str(Path(__file__).resolve().parents[1]))

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {sys.version.split()[0]}")

FileNotFoundError: Cannot find project root. Expected a config/ folder somewhere above this notebook.

## Cell 2 — Load config and set up logging

All constants come from `config/settings.py` — no values are hardcoded in this notebook.  
`logging` gives every message a timestamp, which is useful when a download takes several minutes.

In [ ]:
from config import (
    RAW_DATA_DIR,
    START_YEAR,
    END_YEAR,
    WB_INDICATORS,
    LOG_FORMAT,
    LOG_DATE_FORMAT,
)

# Configure logging — one call here covers all cells below.
# force=True re-applies the config if the kernel has already initialised logging.
logging.basicConfig(
    level=logging.INFO,
    format=LOG_FORMAT,
    datefmt=LOG_DATE_FORMAT,
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("wbg_data")

OUTPUT_FILE = RAW_DATA_DIR / "world_bank_data.csv"

log.info("Config loaded")
log.info("Time period  : %d - %d (%d years)", START_YEAR, END_YEAR, END_YEAR - START_YEAR + 1)
log.info("Indicators   : %d", len(WB_INDICATORS))
log.info("Output file  : %s", OUTPUT_FILE)

print("\nIndicators to download:")
for code, col in WB_INDICATORS.items():
    print(f"  {code:<30}  ->  {col}")

## Cell 3 — Download function

**How `wbgapi` works:**  
`wb.data.fetch(indicator, 'all', time=range(...))` streams observations as a generator of dicts.  
Each dict contains `economy` (ISO3 code), `time` (e.g. `'YR2005'`), and `value`.

We try `fetch()` first. If it fails, we fall back to `wb.data.DataFrame()` which returns a wide  
table (countries as rows, years as columns) that we reshape to long format with `melt()`.

If both methods fail, the function returns `None` and logs an error — the download continues  
for the remaining indicators rather than crashing entirely.

In [ ]:
def fetch_indicator(
    wb_code: str,
    col_name: str,
    start: int,
    end: int,
) -> "pd.DataFrame | None":
    """
    Download one World Bank indicator for all economies over [start, end].

    Returns a long-format DataFrame with columns [country, year, col_name],
    or None if both download methods fail.

    Parameters
    ----------
    wb_code  : World Bank indicator code, e.g. 'NY.GDP.MKTP.CD'
    col_name : Target column name in the output DataFrame
    start    : First year (inclusive)
    end      : Last year (inclusive)
    """

    def clean_year_col(series):
        """Strip 'YR' prefix and convert to integer year."""
        return pd.to_numeric(
            series.astype(str).str.replace("YR", "", regex=False),
            errors="coerce",
        )

    # ── Method 1: wb.data.fetch ───────────────────────────────────────────────
    try:
        records = [
            {
                "country": item["economy"],
                "year":    item["time"],
                col_name:  item["value"],
            }
            for item in wb.data.fetch(wb_code, "all", time=range(start, end + 1))
        ]

        if not records:
            raise ValueError("fetch() returned 0 records")

        df = pd.DataFrame(records)
        df["year"] = clean_year_col(df["year"])
        df = df.dropna(subset=["year", "country"])
        df["year"] = df["year"].astype(int)

        log.info("  [OK ] %-32s %6d rows  (fetch)", col_name, len(df))
        return df

    except Exception as err:
        log.warning("  [WARN] %-30s fetch() failed -- %s", col_name, err)

    # ── Method 2: wb.data.DataFrame (fallback) ────────────────────────────────
    try:
        wide = wb.data.DataFrame(
            wb_code, "all",
            time=range(start, end + 1),
            labels=False,
        ).reset_index()

        year_cols = [c for c in wide.columns if str(c).startswith("YR")]

        if not year_cols:
            raise ValueError("No YR* columns found in DataFrame output")

        df = (
            wide.melt(
                id_vars=["economy"],
                value_vars=year_cols,
                var_name="year",
                value_name=col_name,
            )
            .rename(columns={"economy": "country"})
        )

        df["year"] = clean_year_col(df["year"])
        df = df.dropna(subset=["year", "country"])
        df["year"] = df["year"].astype(int)

        log.info("  [OK ] %-32s %6d rows  (DataFrame fallback)", col_name, len(df))
        return df

    except Exception as err:
        log.error("  [FAIL] %-30s both methods failed -- %s", col_name, err)
        return None


log.info("fetch_indicator() ready.")

## Cell 4 — Run the download

Downloads all 16 indicators one at a time and merges them into a single wide DataFrame.

**Expected runtime:** 3–6 minutes depending on internet speed.  
`time.sleep(0.3)` between calls avoids hitting the World Bank API rate limit.

In [ ]:
log.info("Starting download: %d indicators, %d-%d", len(WB_INDICATORS), START_YEAR, END_YEAR)
t_start = time.time()

downloaded = []   # list of DataFrames, one per indicator
failed     = []   # column names that could not be downloaded

for i, (wb_code, col_name) in enumerate(WB_INDICATORS.items(), start=1):
    log.info("[%2d/%d] %s", i, len(WB_INDICATORS), wb_code)
    result = fetch_indicator(wb_code, col_name, START_YEAR, END_YEAR)

    if result is not None:
        downloaded.append(result)
    else:
        failed.append(col_name)

    time.sleep(0.3)   # respect API rate limit

# Guard: at least one indicator must have succeeded
if not downloaded:
    raise RuntimeError(
        "All indicator downloads failed. "
        "Check your internet connection and that wbgapi is installed."
    )

# Merge all indicators into one wide DataFrame
log.info("Merging %d indicator(s) into one DataFrame...", len(downloaded))
wb_data = downloaded[0]
for df in downloaded[1:]:
    wb_data = wb_data.merge(df, on=["country", "year"], how="outer")

wb_data = wb_data.sort_values(["country", "year"]).reset_index(drop=True)

elapsed = time.time() - t_start
log.info("Download complete in %.1f s.", elapsed)

if failed:
    log.warning("Failed indicators (%d): %s", len(failed), failed)

print(f"\nShape      : {wb_data.shape[0]:,} rows x {wb_data.shape[1]} columns")
print(f"Countries  : {wb_data['country'].nunique()}")
print(f"Years      : {wb_data['year'].min()} - {wb_data['year'].max()}")

## Cell 5 — Data quality report

Completeness check across all indicators for the full dataset.

- **OK (>= 80%)** — reliable for most countries
- **SPARSE (50-79%)** — gaps present; imputation will be needed
- **VERY SPARSE (< 50%)** — limited coverage; review before using in the model

Fiscal indicators (`debt_to_gdp`, `tax_revenue_gdp`, `interest_payments_gdp`) are expected  
to be sparse — government finance data is less consistently reported than GDP.

In [ ]:
indicator_cols = [c for c in wb_data.columns if c not in ("country", "year")]
total_rows     = len(wb_data)

print(f"{'Column':<30} {'Non-null':>9} {'Complete':>9}  Status")
print("-" * 65)

for col in indicator_cols:
    non_null     = wb_data[col].notna().sum()
    completeness = non_null / total_rows * 100
    if completeness >= 80:
        status = "OK"
    elif completeness >= 50:
        status = "SPARSE"
    else:
        status = "VERY SPARSE -- review before modelling"
    print(f"{col:<30} {non_null:>9,} {completeness:>8.1f}%  {status}")

print("-" * 65)
print(f"{'Total rows':<30} {total_rows:>9,}")

## Cell 6 — Preview sample countries

Sanity check: look at the last 5 years for three representative economies  
(one per intended type) to confirm values look plausible before saving.

In [ ]:
# One representative per intended economy type — change as needed
preview_countries = ["USA", "BRA", "NGA"]

recent = wb_data["year"] >= (END_YEAR - 4)   # last 5 years
mask   = wb_data["country"].isin(preview_countries)

preview = (
    wb_data[recent & mask]
    .sort_values(["country", "year"])
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width",       200)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

print(preview.to_string(index=False))

## Cell 7 — Save

Saves the complete file (all ~260 countries) to `data/raw/world_bank_data.csv`.

The raw file is kept general-purpose. Filtering to your dissertation sample  
happens in the next notebook once you have populated `COUNTRIES_BY_TYPE` in  
`config/settings.py`.

In [ ]:
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
wb_data.to_csv(OUTPUT_FILE, index=False)

size_kb = OUTPUT_FILE.stat().st_size / 1024

log.info("Saved   : %s", OUTPUT_FILE)
log.info("Rows    : %d", len(wb_data))
log.info("Columns : %d  ->  %s", len(wb_data.columns), wb_data.columns.tolist())
log.info("Size    : %.1f KB", size_kb)
log.info("")
log.info("Next steps:")
log.info("  1. Review the quality report above to assess coverage by country.")
log.info("  2. Populate COUNTRIES_BY_TYPE in config/settings.py.")
log.info("  3. Run src/notebooks/extract.ipynb.")